### Step 1: Project Setup

In [ ]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY    = os.getenv("GOOGLE_API_KEY")
GROQ_API_KEY      = os.getenv("GROQ_API_KEY")
OPENROUTER_API_KEY      = os.getenv("OPENROUTER_API_KEY")

print("Gemini key loaded:       ", "✅" if GEMINI_API_KEY    else "❌ Missing!")
print("Groq key loaded:         ", "✅" if GROQ_API_KEY      else "❌ Missing!")
print("OpenRouter key loaded:   ", "✅" if OPENROUTER_API_KEY      else "❌ Missing!")

Gemini key loaded:        ✅
Groq key loaded:          ✅
OpenRouter key loaded:    ✅


In [ ]:
from google import genai
from groq import Groq

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
print("✅ GEMINI client ready")

groq_client = Groq(api_key=GROQ_API_KEY)
print("✅ GROQ client ready")

✅ GEMINI client ready
✅ GROQ client ready


### Step 2: Index creation without any LLM

#### PageIndex's underlying Flash function without LLM summary

Instead of running PageIndexClient.submit_document() (which forces LLM summarization on every node), call PageIndex's underlying Flash function directly with summary=False. Using optimize="merge" (deterministic, LLM-free pass) or optimize=False builds the entire hierarchical tree purely from the PDF layout and bookmarks in seconds—with zero Ollama calls and zero API rate limits.

In [ ]:
import json
import time
from pathlib import Path
import fitz
import pymupdf
from pageindex.flash import page_index_flash


def build_fast_legal_index(corpus_dir: str = "./corpus", output_dir: str = "./legal_index",):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    manifest = {}

    for pdf_path in sorted(Path(corpus_dir).glob("*.pdf")):
        t0 = time.perf_counter()
        print(f"[INDEXING] {pdf_path.name} (zero-LLM Flash mode) ...")

        # Pure deterministic layout/bookmark tree - zero LLM calls
        tree = page_index_flash(
            str(pdf_path),
            summary=False,
            optimize="merge",  # use optimize=False if raw unmerged nodes is needed
        )

        # Save hierarchical tree JSON
        tree_file = out / f"{pdf_path.stem}_tree.json"
        tree_file.write_text(json.dumps(tree, indent=2, ensure_ascii=False), encoding="utf-8")

        # Extract & cache 1-indexed verbatim pages once for <5ms retrieval in LangGraph
        doc = fitz.open(pdf_path)
        pages = {str(i + 1): page.get_text("text") for i, page in enumerate(doc)}
        doc.close()

        pages_file = out / f"{pdf_path.stem}_pages.json"
        pages_file.write_text(json.dumps(pages, indent=2, ensure_ascii=False), encoding="utf-8")

        manifest[pdf_path.stem] = {
            "source_pdf": pdf_path.name,
            "page_count": len(pages),
            "tree_file": str(tree_file),
            "pages_file": str(pages_file),
        }
        print(f"[DONE] {pdf_path.name} ({len(pages)} pages) in {time.perf_counter() - t0:.2f}s")

    (out / "manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(f"\nAll statutes indexed. Manifest saved to {out / 'manifest.json'}")


if __name__ == "__main__":
    build_fast_legal_index()

### Step 3: Tree Audit (top level view) for all the documents tree.json files

- bnsAct2023_tree

In [6]:
import json
from pathlib import Path

tree_path = Path("../legal_index/bnsAct2023_tree.json")
data = json.loads(tree_path.read_text(encoding="utf-8"))

print(f"Doc: {data.get('doc_name')} | Title: {data.get('doc_title')}\n")
print("=== TOP-LEVEL NODES ===")
for node in data.get("structure", []):
    children = node.get("nodes", [])
    print(
        f"[{node['node_id']}] pp. {node['start_index']}-{node['end_index']} "
        f"| children={len(children)} | {node['title']}"
    )
    # Preview all children of each non-empty top-level section
    for child in children[:]:
        print(
            f"    └─ [{child['node_id']}] pp. {child['start_index']}-{child['end_index']} "
            f"| {child['title']}"
        )

Doc: bnsAct2023.pdf | Title: Bharatiya Nyaya Sanhita, 2023

=== TOP-LEVEL NODES ===
[0000] pp. 1-2 | children=0 | homepage
[0001] pp. 2-20 | children=18 | index
    └─ [0002] pp. 2-2 | CHAPTER I PRELIMINARY; CHAPTER II OF PUNISHMENTS
    └─ [0003] pp. 2-4 | CHAPTER III GENERAL EXCEPTIONS
    └─ [0004] pp. 4-5 | CHAPTER IV OF ABETMENT, CRIMINAL CONSPIRACY AND ATTEMPT
    └─ [0005] pp. 5-7 | CHAPTER V OF OFFENCES AGAINST WOMAN AND CHILD
    └─ [0006] pp. 7-9 | CHAPTER VI OF OFFENCES AFFECTING THE HUMAN BODY
    └─ [0007] pp. 9-9 | CHAPTER VII OF OFFENCES AGAINST THE STATE
    └─ [0008] pp. 9-10 | CHAPTER VIII OF OFFENCES RELATING TO THE ARMY, NAVY AND AIR FORCE
    └─ [0009] pp. 10-10 | CHAPTER IX OF OFFENCES RELATING TO ELECTIONS
    └─ [0010] pp. 10-11 | CHAPTER X OF OFFENCES RELATING TO COIN, CURRENCY-NOTES, BANK-NOTES, AND GOVERNMENT STAMPS
    └─ [0011] pp. 11-11 | CHAPTER XI OF OFFENCES AGAINST THE PUBLIC TRANQUILLITY
    └─ [0012] pp. 11-12 | CHAPTER XII OF OFFENCES BY OR RELATING

- bnssAct2023_tree

In [7]:
import json
from pathlib import Path

tree_path = Path("../legal_index/bnssAct2023_tree.json")
data = json.loads(tree_path.read_text(encoding="utf-8"))

print(f"Doc: {data.get('doc_name')} | Title: {data.get('doc_title')}\n")
print("=== TOP-LEVEL NODES ===")
for node in data.get("structure", []):
    children = node.get("nodes", [])
    print(
        f"[{node['node_id']}] pp. {node['start_index']}-{node['end_index']} "
        f"| children={len(children)} | {node['title']}"
    )
    # Preview all children of each non-empty top-level section
    for child in children[:]:
        print(
            f"    └─ [{child['node_id']}] pp. {child['start_index']}-{child['end_index']} "
            f"| {child['title']}"
        )
        

Doc: bnssAct2023.pdf | Title: Bharatiya Nagarik Suraksha Sanhita, 2023

=== TOP-LEVEL NODES ===
[0000] pp. 1-4 | children=0 | Preface
[0001] pp. 4-9 | children=0 | CHAPTER II 
[0002] pp. 9-11 | children=0 | CHAPTER III 
[0003] pp. 11-13 | children=0 | CHAPTER IV 
[0004] pp. 13-20 | children=0 | CHAPTER V 
[0005] pp. 20-26 | children=4 | CHAPTER VI 
    └─ [0006] pp. 20-21 | A. —Summons
    └─ [0007] pp. 21-24 | B. —Warrant of arrest
    └─ [0008] pp. 24-26 | C. —Proclamation and attachment
    └─ [0009] pp. 26-26 | D. —Other rules regarding processes
[0010] pp. 26-32 | children=4 | CHAPTER VII 
    └─ [0011] pp. 26-27 | A. —Summons to produce
    └─ [0012] pp. 27-29 | B. —Search-warrants
    └─ [0013] pp. 29-30 | C. —General provisions relating to searches
    └─ [0014] pp. 30-32 | D. —Miscellaneous
[0015] pp. 32-36 | children=0 | CHAPTER VIII 
[0016] pp. 36-42 | children=0 | CHAPTER IX 
[0017] pp. 42-45 | children=0 | CHAPTER X 
[0018] pp. 45-52 | children=4 | CHAPTER XI 
    └─ [0019

- constitutionOfIndia1950

In [8]:
import json
from pathlib import Path

tree_path = Path("../legal_index/constitutionOfIndia1950_tree.json")
data = json.loads(tree_path.read_text(encoding="utf-8"))

print(f"Doc: {data.get('doc_name')} | Title: {data.get('doc_title')}\n")
print("=== TOP-LEVEL NODES ===")
for node in data.get("structure", []):
    children = node.get("nodes", [])
    print(
        f"[{node['node_id']}] pp. {node['start_index']}-{node['end_index']} "
        f"| children={len(children)} | {node['title']}"
    )
    # Preview all children of each non-empty top-level section
    for child in children[:]:
        print(
            f"    └─ [{child['node_id']}] pp. {child['start_index']}-{child['end_index']} "
            f"| {child['title']}"
        )
        

Doc: constitutionOfIndia1950.pdf | Title: भारत का संविधान

=== TOP-LEVEL NODES ===
[0000] pp. 1-1 | children=0 | Preface
[0001] pp. 2-2 | children=0 | PREFACE
[0002] pp. 3-4 | children=0 | LIST OF ABBREVIATIONS USED
[0003] pp. 5-6 | children=0 | PART III
[0004] pp. 6-7 | children=0 | PART IV
[0005] pp. 7-7 | children=0 | PART IVA
[0006] pp. 7-12 | children=6 | PART V
    └─ [0007] pp. 7-8 | The President and Vice-President
    └─ [0008] pp. 8-8 | Council of Ministers; The Attorney-General for India
    └─ [0009] pp. 9-9 | Conduct of Government Business
    └─ [0010] pp. 9-11 | CHAPTER II .⎯PARLIAMENT
    └─ [0011] pp. 11-11 | CHAPTER III .⎯LEGISLATIVE POWERS OF THE PRESIDENT
    └─ [0012] pp. 11-12 | CHAPTER IV .⎯ THE UNION JUDICIARY
[0013] pp. 12-13 | children=0 | PART VI
[0014] pp. 13-13 | children=0 | CHAPTER II .⎯ THE EXECUTIVE
[0015] pp. 13-17 | children=5 | CHAPTER III .⎯THE STATE LEGISLATURE
    └─ [0016] pp. 13-14 | General
    └─ [0017] pp. 14-14 | Officers of the State Legisl

- dpdpAct2023

In [12]:
import json
from pathlib import Path

tree_path = Path("../legal_index/dpdpAct2023_tree.json")
data = json.loads(tree_path.read_text(encoding="utf-8"))

print(f"Doc: {data.get('doc_name')} | Title: {data.get('doc_title')}\n")
print("=== TOP-LEVEL NODES ===")
for node in data.get("structure", []):
    children = node.get("nodes", [])
    print(
        f"[{node['node_id']}] pp. {node['start_index']}-{node['end_index']} "
        f"| children={len(children)} | {node['title']}"
    )
    # Preview all children of each non-empty top-level section
    for child in children[:]:
        print(
            f"    └─ [{child['node_id']}] pp. {child['start_index']}-{child['end_index']} "
            f"| {child['title']}"
        )
        

Doc: dpdpAct2023.pdf | Title: Digital Personal Data Protection (DPDP) Act, 2023

=== TOP-LEVEL NODES ===
[0000] pp. 1-4 | children=0 | Preface
[0001] pp. 4-9 | children=3 | CHAPTER II 
    └─ [0002] pp. 4-6 | OBLIGATIONS OF DATA FIDUCIARY
    └─ [0003] pp. 6-8 | Illustrations.
    └─ [0004] pp. 8-9 | Illustrations.
[0005] pp. 9-10 | children=0 | CHAPTER III 
[0006] pp. 11-12 | children=0 | CHAPTER IV 
[0007] pp. 12-14 | children=0 | CHAPTER V 
[0008] pp. 14-15 | children=0 | CHAPTER VI 
[0009] pp. 15-16 | children=0 | CHAPTER VII 
[0010] pp. 16-17 | children=0 | CHAPTER VIII 
[0011] pp. 17-20 | children=0 | CHAPTER IX 
[0012] pp. 21-21 | children=0 | THE SCHEDULE


- itActAct2020

In [13]:
import json
from pathlib import Path

tree_path = Path("../legal_index/itAct2020_tree.json")
data = json.loads(tree_path.read_text(encoding="utf-8"))

print(f"Doc: {data.get('doc_name')} | Title: {data.get('doc_title')}\n")
print("=== TOP-LEVEL NODES ===")
for node in data.get("structure", []):
    children = node.get("nodes", [])
    print(
        f"[{node['node_id']}] pp. {node['start_index']}-{node['end_index']} "
        f"| children={len(children)} | {node['title']}"
    )
    # Preview all children of each non-empty top-level section
    for child in children[:]:
        print(
            f"    └─ [{child['node_id']}] pp. {child['start_index']}-{child['end_index']} "
            f"| {child['title']}"
        )
        

Doc: itAct2020.pdf | Title: Information Technology Act, 2000 (IT Act)

=== TOP-LEVEL NODES ===
[0000] pp. 1-2 | children=0 | Preface
[0001] pp. 2-2 | children=0 | CHAPTER VII ; ELECTRONIC SIGNATURE CERTIFICATES; CHAPTER VIII ; DUTIES OF SUBSCRIBERS; CHAPTER IX ; PENALTIES, COMPENSATION AND ADJUDICATION
[0002] pp. 2-3 | children=0 | CHAPTER X 
[0003] pp. 3-3 | children=0 | p.3
[0004] pp. 3-5 | children=0 | CHAPTER XIII MISCELLANEOUS
[0005] pp. 5-8 | children=0 | CHAPTER 1 PRELIMINARY
[0006] pp. 8-9 | children=0 | CHAPTER II 
[0007] pp. 9-11 | children=0 | CHAPTER III 
[0008] pp. 11-12 | children=0 | CHAPTER IV 
[0009] pp. 12-12 | children=0 | CHAPTER V 
[0010] pp. 12-23 | children=5 | CHAPTER VI 
    └─ [0011] pp. 12-16 | REGULATION OF CERTIFYING AUTHORITIES
    └─ [0012] pp. 16-18 | 1 [ELECTRONIC SIGNATURE] CERTIFICATES
    └─ [0013] pp. 18-18 | CHAPTER VIII 
    └─ [0014] pp. 18-21 | CHAPTER IX 
    └─ [0015] pp. 21-23 | CHAPTER X 
[0016] pp. 23-30 | children=0 | CHAPTER XI 
[0017] pp

### Step 4: Get Summary to the nodes of JSON

Only the root nodes will be generating the summary from LLM based on:
- node title
- node information stores in page.json

In [ ]:
# Count the total no. of summaries that will be generated before actually generating it. (to save time)
# To actually generate summaries for nodes
# For long sections, it was using only the few page from start, but that summary will be incomplete.
# tree nodes already contain all sub-section titles inside node["nodes"] and node["key_items"]. 
# Feeding those sub-headings plus the opening page text gives the LLM visibility across all 25 pages while staying well within Groq's 8,000 TPM limit

import json
import re
import time
from pathlib import Path
import litellm
from dotenv import load_dotenv

load_dotenv()

MODEL = "groq/openai/gpt-oss-20b"   # Free tier: 30 RPM, 8K TPM, 1K RPD
DRY_RUN = False                     # 1) Run True to check counts, 2) Set False to generate
CONSTITUTION_BODY_START = 32        # actual start

SKIP_KEYWORDS = ("homepage", "index", "table of contents", "contents", "preface", "abbreviations")
MIN_BODY_PAGE = {"constitutionOfIndia1950": CONSTITUTION_BODY_START}

def should_skip(node: dict, doc_key: str) -> bool:
    title = node.get("title", "").strip().lower()
    start_p = int(node.get("start_index") or 1)
    if any(k in title for k in SKIP_KEYWORDS):
        return True
    if re.match(r"^form\s+no\.?", title):  # Skip BNSS Second Schedule forms
        return True
    return start_p < MIN_BODY_PAGE.get(doc_key, 1)

def get_summary_targets(structure: list, doc_key: str) -> list:
    targets = []
    for node in structure:
        if should_skip(node, doc_key):
            continue
        title = node.get("title", "").strip().lower()
        span = int(node.get("end_index") or 1) - int(node.get("start_index") or 1)
        if node.get("nodes") and span > 40 and title != "compare":
            targets.extend(c for c in node["nodes"] if not should_skip(c, doc_key))
        else:
            targets.append(node)
    return targets


litellm.drop_params = True

def summarize(title: str, text: str) -> str:
    resp = litellm.completion(
        model=MODEL,
        temperature=0,
        max_tokens=512,
        reasoning_effort="low",
        num_retries=5,
        messages=[{
            "role": "user",
            "content": (
                f"Write a concise 1-2 sentence plain-text summary of this Indian legal section ('{title}'), "
                f"listing key topics, offences, procedures, or rights:\n\n{text[:2500]}"
            ),
        }],
    )
    out = (resp.choices[0].message.content or "").strip()
    return re.sub(r"<think>.*?</think>", "", out, flags=re.S).strip()

def build_summary_context(node: dict, pages: dict, max_chars: int = 2500) -> str:
    # Collect immediate child titles / key_items so long chapters aren't blind to later sections
    subtopics = []
    for c in node.get("nodes", []):
        if c.get("title") and not c["title"].startswith("p."):
            subtopics.append(c["title"])
        subtopics.extend(k for k in c.get("key_items", []) if not k.startswith("p."))
    subtopics.extend(k for k in node.get("key_items", []) if not k.startswith("p."))

    header = f"Sub-sections in this chapter: {'; '.join(dict.fromkeys(subtopics))[:1200]}\n\n" if subtopics else ""

    s = int(node.get("start_index") or 1)
    e = int(node.get("end_index") or s)
    body = "\n".join(pages.get(str(p), "") for p in range(s, e + 1))
    return (header + body)[:max_chars]

def add_summaries(index_dir: str = "./legal_index"):
    base = Path(index_dir)
    manifest = json.loads((base / "manifest.json").read_text(encoding="utf-8"))

    for doc_key, meta in manifest.items():
        tree_path = Path(meta["tree_file"])
        tree = json.loads(tree_path.read_text(encoding="utf-8"))
        pages = json.loads(Path(meta["pages_file"]).read_text(encoding="utf-8"))

        targets = get_summary_targets(tree.get("structure", []), doc_key)
        for n in targets:
            s = n.get("summary", "").strip()
            if not s or not s.endswith("."):
                n.pop("summary", None)

        pending = [n for n in targets if not n.get("summary")]
        print(f"{doc_key}: {len(targets)} targets ({len(pending)} remaining)")

        if DRY_RUN or not pending:
            continue

        for i, node in enumerate(pending, 1):
            s = int(node.get("start_index") or 1)
            e = int(node.get("end_index") or s)
            # snippet = "\n".join(pages.get(str(p), "") for p in range(s, e + 1))
            snippet = build_summary_context(node, pages) # every chapter summary will then cover all topics from its first page to its last page
            if snippet.strip():
                node["summary"] = summarize(node.get("title", ""), snippet)
                print(f"  [{i}/{len(pending)}] {node['node_id']}: {node['summary'][:80]}...")
                time.sleep(6.5)

        tree_path.write_text(json.dumps(tree, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"  -> Saved {tree_path.name}")

if __name__ == "__main__":
    add_summaries()

# Output:

# bnsAct2023: 20 summary targets
# bnssAct2023: 84 summary targets
# constitutionOfIndia1950: 68 summary targets
# dpdpAct2023: 10 summary targets
# itAct2020: 13 summary targets

# bnsAct2023: 20 targets (20 remaining)
# bnssAct2023: 26 targets (26 remaining)
# constitutionOfIndia1950: 43 targets (43 remaining)
# dpdpAct2023: 9 targets (9 remaining)
# itAct2020: 12 targets (12 remaining)

### Step 5: Tree Combining

#### Problem with this approach

Combining all five of the indexed Acts (bnsAct2023, bnssAct2023, constitutionOfIndia1950, dpdpAct2023, and itAct2020) requires solving two issues that single-document tutorials skip:   
- node_id collisions: Every *_tree.json starts at "0000". Merging them without namespacing (doc_key::node_id, e.g., bnsAct2023::0148) causes cross-Act lookup collisions.
- Token overflow: Passing all five complete trees into one LLM call blows past free-tier context/TPM limits. 

**Before merging lets view the tree nodes**

#### View the Headings with Page Number from the JSON tree for [Constitution, BNSS]

In [ ]:
import json
from pathlib import Path
import re

CONSTITUTION_BODY_START = 32        # actual start
SKIP_KEYWORDS = ("homepage", "index", "table of contents", "contents", "preface", "abbreviations")
MIN_BODY_PAGE = {"constitutionOfIndia1950": CONSTITUTION_BODY_START}

def should_skip(node: dict, doc_key: str) -> bool:
    title = node.get("title", "").strip().lower()
    start_p = int(node.get("start_index") or 1)
    if any(k in title for k in SKIP_KEYWORDS):
        return True
    if re.match(r"^form\s+no\.?", title):
        return True
    return start_p < MIN_BODY_PAGE.get(doc_key, 1)

def get_summary_targets(structure: list, doc_key: str) -> list:
    targets = []
    for node in structure:
        if should_skip(node, doc_key):
            continue
        title = node.get("title", "").strip().lower()
        span = int(node.get("end_index") or 1) - int(node.get("start_index") or 1)
        if node.get("nodes") and span > 40 and title != "compare":
            targets.extend(c for c in node["nodes"] if not should_skip(c, doc_key))
        else:
            targets.append(node)
    return targets

for doc in ["bnssAct2023", "constitutionOfIndia1950"]:
    tree = json.loads(Path(f"./legal_index/{doc}_tree.json").read_text(encoding="utf-8"))
    targets = get_summary_targets(tree.get("structure", []))
    print(f"\n=== {doc} ({len(targets)} targets) ===")
    for t in targets[:20]:
        print(f"  [{t['node_id']}] pp. {t.get('start_index')}-{t.get('end_index')} | {t.get('title')}")
    if len(targets) > 20:
        print(f"  ... and {len(targets) - 20} more (last: {targets[-1].get('title')})")


=== bnssAct2023 (84 targets) ===
  [0000] pp. 1-4 | Preface
  [0001] pp. 4-9 | CHAPTER II 
  [0002] pp. 9-11 | CHAPTER III 
  [0003] pp. 11-13 | CHAPTER IV 
  [0004] pp. 13-20 | CHAPTER V 
  [0005] pp. 20-26 | CHAPTER VI 
  [0010] pp. 26-32 | CHAPTER VII 
  [0015] pp. 32-36 | CHAPTER VIII 
  [0016] pp. 36-42 | CHAPTER IX 
  [0017] pp. 42-45 | CHAPTER X 
  [0018] pp. 45-52 | CHAPTER XI 
  [0023] pp. 52-52 | CHAPTER XII 
  [0024] pp. 52-65 | CHAPTER XIII 
  [0025] pp. 65-67 | CHAPTER XIV 
  [0026] pp. 67-73 | CHAPTER XV 
  [0027] pp. 73-74 | CHAPTER XVI 
  [0028] pp. 74-76 | CHAPTER XVII 
  [0029] pp. 76-82 | CHAPTER XVIII 
  [0032] pp. 82-84 | CHAPTER XIX 
  [0034] pp. 84-85 | A. —Cases instituted on a police report
  ... and 64 more (last: FORM No. 58)

=== constitutionOfIndia1950 (68 targets) ===
  [0000] pp. 1-1 | Preface
  [0001] pp. 2-2 | PREFACE
  [0002] pp. 3-4 | LIST OF ABBREVIATIONS USED
  [0003] pp. 5-6 | PART III
  [0004] pp. 6-7 | PART IV
  [0005] pp. 7-7 | PART IVA
  [0006

#### View the Child Nodes from the JSON file

In [3]:
# View the Child Nodes from the JSON file

import json
from pathlib import Path

tree = json.loads(Path("./legal_index/constitutionOfIndia1950_tree.json").read_text(encoding="utf-8"))
for n in tree.get("structure", []):
    print(f"[{n['node_id']}] pp. {n.get('start_index')}-{n.get('end_index')} | children={len(n.get('nodes', []))} | {n.get('title')}")

tree = json.loads(Path("./legal_index/bnssAct2023_tree.json").read_text(encoding="utf-8"))
for n in tree.get("structure", []):
    print(f"[{n['node_id']}] pp. {n.get('start_index')}-{n.get('end_index')} | children={len(n.get('nodes', []))} | {n.get('title')}")

tree = json.loads(Path("./legal_index/bnsAct2023_tree.json").read_text(encoding="utf-8"))
for n in tree.get("structure", []):
    print(f"[{n['node_id']}] pp. {n.get('start_index')}-{n.get('end_index')} | children={len(n.get('nodes', []))} | {n.get('title')}")

tree = json.loads(Path("./legal_index/dpdpAct2023_tree.json").read_text(encoding="utf-8"))
for n in tree.get("structure", []):
    print(f"[{n['node_id']}] pp. {n.get('start_index')}-{n.get('end_index')} | children={len(n.get('nodes', []))} | {n.get('title')}")

tree = json.loads(Path("./legal_index/itAct2020_tree.json").read_text(encoding="utf-8"))
for n in tree.get("structure", []):
    print(f"[{n['node_id']}] pp. {n.get('start_index')}-{n.get('end_index')} | children={len(n.get('nodes', []))} | {n.get('title')}")

[0000] pp. 1-1 | children=0 | Preface
[0001] pp. 2-2 | children=0 | PREFACE
[0002] pp. 3-4 | children=0 | LIST OF ABBREVIATIONS USED
[0003] pp. 5-6 | children=0 | PART III
[0004] pp. 6-7 | children=0 | PART IV
[0005] pp. 7-7 | children=0 | PART IVA
[0006] pp. 7-12 | children=6 | PART V
[0013] pp. 12-13 | children=0 | PART VI
[0014] pp. 13-13 | children=0 | CHAPTER II .⎯ THE EXECUTIVE
[0015] pp. 13-17 | children=5 | CHAPTER III .⎯THE STATE LEGISLATURE
[0021] pp. 17-17 | children=0 | CHAPTER VI .⎯SUBORDINATE COURTS; PART VIII
[0022] pp. 17-18 | children=0 | PART IX
[0023] pp. 18-19 | children=0 | PART IXA
[0024] pp. 19-20 | children=0 | PART X
[0025] pp. 20-21 | children=0 | PART XI
[0026] pp. 21-23 | children=0 | PART XII
[0027] pp. 23-23 | children=0 | PART XIII
[0028] pp. 23-25 | children=0 | PART XIV
[0029] pp. 25-25 | children=0 | PART XV
[0030] pp. 25-26 | children=0 | PART XVI
[0031] pp. 26-27 | children=0 | PART XVII
[0032] pp. 27-28 | children=0 | PART XVIII
[0033] pp. 28-30 | c

In [2]:
import json
from pathlib import Path

tree = json.loads(Path("./legal_index/bnsAct2023_tree.json").read_text(encoding="utf-8"))
for n in tree.get("structure", []):
    print(f"[{n['node_id']}] pp. {n.get('start_index')}-{n.get('end_index')} | children={len(n.get('nodes', []))} | {n.get('title')}")

[0000] pp. 1-2 | children=0 | homepage
[0001] pp. 2-20 | children=18 | index
[0024] pp. 20-74 | children=24 | Compare
[0067] pp. 74-237 | children=19 | chapters


#### Merging the JSON files using Namespace

In [ ]:
# Merge all the index json files to generate a single JSON file

import json
from pathlib import Path
import re

CONSTITUTION_BODY_START = 32        # actual start
SKIP_KEYWORDS = ("homepage", "index", "table of contents", "contents", "preface", "abbreviations")
MIN_BODY_PAGE = {"constitutionOfIndia1950": CONSTITUTION_BODY_START}

def should_skip(node: dict, doc_key: str) -> bool:
    title = node.get("title", "").strip().lower()
    start_p = int(node.get("start_index") or 1)
    if any(k in title for k in SKIP_KEYWORDS):
        return True
    if re.match(r"^form\s+no\.?", title):
        return True
    return start_p < MIN_BODY_PAGE.get(doc_key, 1)

def prefix_ids(nodes: list, doc_key: str) -> list:
    out = []
    for node in nodes:
        if should_skip(node, doc_key):
            continue
        node["node_id"] = f"{doc_key}::{node['node_id']}"
        if node.get("nodes"):
            node["nodes"] = prefix_ids(node["nodes"], doc_key)
        out.append(node)
    return out

CONSTITUTION_BODY_START = 32       # actual start

SKIP_KEYWORDS = ("homepage", "index", "table of contents", "contents", "preface", "abbreviations")
MIN_BODY_PAGE = {"constitutionOfIndia1950": CONSTITUTION_BODY_START}

def should_skip(node: dict, doc_key: str) -> bool:
    title = node.get("title", "").strip().lower()
    start_p = int(node.get("start_index") or 1)
    if any(k in title for k in SKIP_KEYWORDS):
        return True
    if re.match(r"^form\s+no\.?", title):  # Skip BNSS Second Schedule forms
        return True
    return start_p < MIN_BODY_PAGE.get(doc_key, 1)


def merge_trees(index_dir: str = "./legal_index"):
    base = Path(index_dir)
    manifest = json.loads((base / "manifest.json").read_text(encoding="utf-8"))
    merged = []

    for doc_key, meta in manifest.items():
        tree = json.loads(Path(meta["tree_file"]).read_text(encoding="utf-8"))
        merged.append({
            "node_id": f"{doc_key}::ROOT",
            "title": tree.get("doc_title") or doc_key,
            "start_index": 1,
            "end_index": meta["page_count"],
            "nodes": prefix_ids(tree.get("structure", []), doc_key),
        })

    out_path = base / "merged_tree.json"
    out_path.write_text(json.dumps(merged, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Merged {len(merged)} Acts -> {out_path}")

if __name__ == "__main__":
    merge_trees()

#### Why this Objection Actually Strengthens Your Research
Instead of hiding the LLM-router dependency, make it your central ablation study. In your paper or technical report, compare four configurations on the same 50–100 legal query benchmark:

- Baseline A: Naive Vector RAG (fixed chunking + embeddings).

- Baseline B: Pure LLM Tree Routing (PageIndex-style only—your current multi_doc_rag.py).

- Variant C: Hybrid Vectorless (LLM Tree Router + SQLite FTS5/BM25 Union + Deterministic Cross-Ref Expansion).

- Full System (Ours): Variant C + Cyclic LangGraph Completeness Loop & Fail-Closed Verifier.

Measuring Statutory Recall, Proviso/Exception Capture Rate, Cross-Act Recall, and Verbatim Citation Faithfulness across B → C → Full System proves empirically both why separating retrieval from generation prevents hallucination and why deterministic recall guards are necessary to prevent LLM routing omissions.


### Step 6: View the Merged JSON Tree

Those summary=✗ nodes are not missing summaries—their summaries were generated one level deeper on their child nodes because of the span > 40 rule in get_summary_targets.py:

- bnsAct2023::0067 (chapters, pp. 74–237): span = 163 > 40, so all 19 individual chapters inside it (0068 through 0304) received summaries instead of the outer wrapper.

- bnssAct2023::0033 (CHAPTER XX, pp. 84–127): span = 43 > 40, so its sub-chapters (Trial Before a Court of Session, Trial of Warrant-Cases, etc.) were summarized individually.

- constitutionOfIndia1950::0048 (PART V, pp. 57–100): span = 43 > 40, so its chapters (The Executive, Parliament, The Union Judiciary, CAG) received separate summaries.

- constitutionOfIndia1950::0122 (SIXTH SCHEDULE, pp. 311–382): span = 71 > 40, so its sub-nodes were summarized individually.

In [20]:
import json
from pathlib import Path

merged_path = Path("../legal_index/merged_tree.json")
data = json.loads(merged_path.read_text(encoding="utf-8"))

def rollup_summaries(nodes: list):
    for node in nodes:
        children = node.get("nodes", [])
        if children:
            rollup_summaries(children)
            if not node.get("summary"):
                child_titles = [c["title"].strip() for c in children if c.get("title")]
                child_sums = [c["summary"].strip() for c in children if c.get("summary")]
                if child_sums:
                    node["summary"] = f"Covers {', '.join(child_titles[:6])}. " + " ".join(child_sums[:2])

rollup_summaries(data)
merged_path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
print("Rolled up child summaries into multi-page parent nodes.")

Rolled up child summaries into multi-page parent nodes.


In [19]:
# Top level view with Summary

import json
from pathlib import Path

tree_path = Path("../legal_index/merged_tree.json")
data = json.loads(tree_path.read_text(encoding="utf-8"))  # list of Act root dicts

print(f"Merged Acts: {len(data)}\n")
print("=== TOP-LEVEL NODES ===")
for act in data:
    chapters = act.get("nodes", [])
    print(
        f"[{act['node_id']}] pp. {act['start_index']}-{act['end_index']} "
        f"| children={len(chapters)} | {act['title']}"
    )
    for child in chapters:
        has_summary = "✓" if child.get("summary") else "✗"
        print(
            f"    └─ [{child['node_id']}] pp. {child['start_index']}-{child['end_index']} "
            f"| summary={has_summary} | {child['title']}"
        )

Merged Acts: 5

=== TOP-LEVEL NODES ===
[bnsAct2023::ROOT] pp. 1-237 | children=2 | Bharatiya Nyaya Sanhita, 2023
    └─ [bnsAct2023::0024] pp. 20-74 | summary=✓ | Compare
    └─ [bnsAct2023::0067] pp. 74-237 | summary=✓ | chapters
[bnssAct2023::ROOT] pp. 1-249 | children=24 | Bharatiya Nagarik Suraksha Sanhita, 2023
    └─ [bnssAct2023::0001] pp. 4-9 | summary=✓ | CHAPTER II 
    └─ [bnssAct2023::0002] pp. 9-11 | summary=✓ | CHAPTER III 
    └─ [bnssAct2023::0003] pp. 11-13 | summary=✓ | CHAPTER IV 
    └─ [bnssAct2023::0004] pp. 13-20 | summary=✓ | CHAPTER V 
    └─ [bnssAct2023::0005] pp. 20-26 | summary=✓ | CHAPTER VI 
    └─ [bnssAct2023::0010] pp. 26-32 | summary=✓ | CHAPTER VII 
    └─ [bnssAct2023::0015] pp. 32-36 | summary=✓ | CHAPTER VIII 
    └─ [bnssAct2023::0016] pp. 36-42 | summary=✓ | CHAPTER IX 
    └─ [bnssAct2023::0017] pp. 42-45 | summary=✓ | CHAPTER X 
    └─ [bnssAct2023::0018] pp. 45-52 | summary=✓ | CHAPTER XI 
    └─ [bnssAct2023::0023] pp. 52-52 | summary=✓ | C

#### View complete tree with summary[:90]

In [21]:
# View complete tree with summary[:90]

import json

with open("../legal_index/merged_tree.json", "r", encoding="utf-8") as f:
    trees = json.load(f)  # list of Act root dicts

def print_tree(node, indent=0):
    prefix = "  " * indent
    node_id = node.get("node_id") or node.get("id") or ""
    title = node.get("title") or node.get("name") or node.get("act_name") or "Untitled"
    summary = (node.get("summary") or "").strip().replace("\n", " ")
    summary_preview = f" — {summary[:90]}..." if len(summary) > 90 else (f" — {summary}" if summary else "")
    print(f"{prefix}├── [{node_id}] {title}{summary_preview}")

    children = node.get("children") or node.get("nodes") or node.get("sections") or []
    for child in children:
        print_tree(child, indent + 1)

print(f"Total Act trees in merged_tree.json: {len(trees)}\n")
for act_tree in trees:
    print_tree(act_tree)

Total Act trees in merged_tree.json: 5

├── [bnsAct2023::ROOT] Bharatiya Nyaya Sanhita, 2023 — Covers Compare, chapters. The section outlines the structure and scope of the Indian Penal...
  ├── [bnsAct2023::0024] Compare — The section outlines the structure and scope of the Indian Penal Code, detailing prelimina...
    ├── [bnsAct2023::0025] Compare (intro)
    ├── [bnsAct2023::0026] CHAPTER II – OF PUNISHMENTS
    ├── [bnsAct2023::0027] CHAPTER III - OF PUNISHMENTS
    ├── [bnsAct2023::0028] CHAPTER III 
    ├── [bnsAct2023::0029] CHAPTER IV 
    ├── [bnsAct2023::0030] CHAPTER IV 
    ├── [bnsAct2023::0031] CHAPTER V 
    ├── [bnsAct2023::0032] CHAPTER V 
    ├── [bnsAct2023::0033] CHAPTER XX 
    ├── [bnsAct2023::0034] CHAPTER VI 
      ├── [bnsAct2023::0035] New Section
      ├── [bnsAct2023::0036] CHAPTER XVI 
        ├── [bnsAct2023::0037] CHAPTER XVI (intro)
        ├── [bnsAct2023::0038] Of wrongful restraint and wrongful confinement
        ├── [bnsAct2023::0039] Of kidnapping

### Step 7: Vectorless RAG Pipeline

Working but time consuming

In [ ]:
import json
import re
import sqlite3
import time
from pathlib import Path
from typing import Any, Literal, TypedDict

import litellm
from dotenv import load_dotenv
from langgraph.graph import END, START, StateGraph
from litellm import Router

load_dotenv()
litellm.drop_params = True  # safely strips reasoning_effort on non-reasoning fallback models

INDEX_DIR = Path("./legal_index") if Path("./legal_index").exists() else Path("../legal_index")

# Actual statutory body start pages (skips ToC and BNS Compare table pp. 20-73)
# STATUTE_START_PAGE = {
#     "bnsAct2023": 74,
#     "constitutionOfIndia1950": 32,
#     "itAct2020": 5,
# }


# ── 1. Multi-Provider Router with Cooldowns ──────────────────────────────────

LLM_ROUTER = Router(
    model_list=[
        {
            "model_name": "legal-primary",
            "litellm_params": {
                "model": "gemini/gemini-3.5-flash-lite",
                "reasoning_effort": "low",
                # "timeout": 12,
            },
        },
        {
            "model_name": "legal-groq-fast",
            "litellm_params": {
                "model": "groq/openai/gpt-oss-20b",
                # "timeout": 12,
            },
        },
        {
            "model_name": "legal-local-ollama",
            "litellm_params": {
                "model": "ollama_chat/qwen2.5:3b",
                "api_base": "http://localhost:11434",
                # "timeout": 12,
            },
        },
    ],
    fallbacks=[{"legal-primary": ["legal-groq-fast", "legal-local-ollama"]}],
    num_retries=1,
    allowed_fails=2,     # cool down primary deployment after 2 consecutive errors/429s
    cooldown_time=60,    # route straight to backup for 60s before probing primary again
    set_verbose=False,
)


def call_router_json(prompt: str, step_name: str, max_tokens: int = 300) -> tuple[dict, dict]:
    """Lightweight JSON call strictly for routing and verification flags."""
    t0 = time.perf_counter()
    resp = LLM_ROUTER.completion(
        model="legal-primary",
        temperature=0,
        max_tokens=max_tokens,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}],
    )
    latency_ms = round((time.perf_counter() - t0) * 1000, 1)
    served_by = getattr(resp, "model", "unknown")
    tokens = getattr(getattr(resp, "usage", None), "total_tokens", 0) or 0

    raw = re.sub(r"<think>.*?</think>", "", resp.choices[0].message.content or "{}", flags=re.S).strip()
    match = re.search(r"\{.*\}", raw, flags=re.S)
    try:
        parsed = json.loads(match.group(0) if match else raw)
    except json.JSONDecodeError:
        parsed = {}

    telemetry = {
        "step": step_name,
        "served_by": served_by,
        "latency_ms": latency_ms,
        "tokens": tokens,
    }
    return parsed, telemetry


def call_router_text(prompt: str, step_name: str, max_tokens: int = 1200) -> tuple[str, dict]:
    """Plain-text synthesis call so legal answers never truncate inside JSON."""
    t0 = time.perf_counter()
    resp = LLM_ROUTER.completion(
        model="legal-primary",
        temperature=0,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    latency_ms = round((time.perf_counter() - t0) * 1000, 1)
    served_by = getattr(resp, "model", "unknown")
    tokens = getattr(getattr(resp, "usage", None), "total_tokens", 0) or 0

    text = re.sub(r"<think>.*?</think>", "", resp.choices[0].message.content or "", flags=re.S).strip()
    telemetry = {
        "step": step_name,
        "served_by": served_by,
        "latency_ms": latency_ms,
        "tokens": tokens,
    }
    return text, telemetry


# ── 2. Deterministic Corpus & In-Memory FTS5 Lexical Index ───────────────────

class LegalCorpusStore:
    def __init__(self, index_dir: Path = INDEX_DIR):
        self.merged_tree: list[dict] = json.loads(
            (index_dir / "merged_tree.json").read_text(encoding="utf-8")
        )
        manifest = json.loads((index_dir / "manifest.json").read_text(encoding="utf-8"))
        self.pages: dict[str, dict[str, str]] = {
            doc_key: json.loads((index_dir / Path(meta["pages_file"]).name).read_text(encoding="utf-8"))
            for doc_key, meta in manifest.items()
        }
        self.node_map: dict[str, dict] = {}
        self._index_nodes(self.merged_tree)
        self.fts_conn = self._build_fts5()

    def _index_nodes(self, nodes: list[dict]) -> None:
        for n in nodes:
            self.node_map[n["node_id"]] = n
            if n.get("nodes"):
                self._index_nodes(n["nodes"])

    def _build_fts5(self) -> sqlite3.Connection:
        conn = sqlite3.connect(":memory:")
        conn.execute("CREATE VIRTUAL TABLE pages_fts USING fts5(doc_key, page_num UNINDEXED, content);")
        rows = [
            (doc_key, str(p), text)
            for doc_key, doc_pages in self.pages.items()
            for p, text in doc_pages.items()
        ]
        conn.executemany("INSERT INTO pages_fts VALUES (?, ?, ?);", rows)
        conn.commit()
        return conn

    def build_clean_catalog(self) -> list[dict]:
        """Unwrap container nodes ('chapters') and skip 'Compare' so all real chapters are visible."""
        catalog = []
        for act in self.merged_tree:
            chapters = []
            for ch in act.get("nodes", []):
                title_low = ch.get("title", "").strip().lower()
                if title_low == "compare":
                    continue
                routable_nodes = ch.get("nodes", []) if title_low == "chapters" else [ch]
                for node in routable_nodes:
                    entry: dict[str, Any] = {"node_id": node["node_id"], "title": node["title"]}
                    if node.get("summary"):
                        entry["summary"] = node["summary"]
                    elif node.get("nodes"):
                        entry["subtopics"] = [c["title"] for c in node["nodes"][:6]]
                    chapters.append(entry)
            catalog.append({"act": act["title"], "chapters": chapters})
        return catalog

    def lexical_search(
        self,
        query: str,
        preferred_docs: set[str] | None = None,
        top_k: int = 4,
    ) -> list[tuple[str, int]]:
        """BM25 + exact section-number lookup, restricted to real statutory body pages."""
        hits: list[tuple[str, int]] = []

        def is_valid(doc_key: str, p: int) -> bool:
            # if p < STATUTE_START_PAGE.get(doc_key, 1):
            #     return False
            if preferred_docs and doc_key not in preferred_docs:
                return False
            return True

        # 1. Exact section/article header match (e.g., "103." or "38.")
        sec_nums = re.findall(r"\b(?:section|sec\.?|article|art\.?)\s*(\d+[A-Z]?)\b", query, flags=re.I)
        for num in dict.fromkeys(sec_nums):
            cur = self.fts_conn.execute(
                "SELECT doc_key, page_num FROM pages_fts WHERE content MATCH ? LIMIT 8;",
                (f'"{num}."',),
            )
            for doc_key, p in cur.fetchall():
                p_int = int(p)
                if is_valid(doc_key, p_int) and (doc_key, p_int) not in hits:
                    hits.append((doc_key, p_int))

        # 2. Keyword BM25 fallback
        stopwords = {
            "the", "and", "for", "under", "what", "how", "when", "with",
            "that", "from", "section", "sections", "act", "police", "legal",
        }
        tokens = [t for t in re.findall(r"[a-z0-9]{3,}", query.lower()) if t not in stopwords]
        if tokens:
            fts_query = " OR ".join(f'"{t}"' for t in dict.fromkeys(tokens[:10]))
            cur = self.fts_conn.execute(
                "SELECT doc_key, page_num FROM pages_fts WHERE pages_fts MATCH ? ORDER BY bm25(pages_fts) LIMIT 15;",
                (fts_query,),
            )
            for doc_key, p in cur.fetchall():
                p_int = int(p)
                if is_valid(doc_key, p_int) and (doc_key, p_int) not in hits:
                    hits.append((doc_key, p_int))
                if len(hits) >= top_k:
                    break

        return hits[:top_k]

    def zoom_to_leaf_nodes(self, branch_ids: list[str], query: str, max_leaves: int = 3) -> list[str]:
        """Deterministically drill routed chapters down to top matching leaf nodes (0 LLM calls)."""
        stopwords = {"the", "and", "for", "under", "what", "how", "when", "with", "that", "from"}
        q_tokens = {t for t in re.findall(r"[a-z0-9]{3,}", query.lower()) if t not in stopwords}

        def resolve(nid: str) -> list[str]:
            node = self.node_map.get(nid)
            if not node:
                return []
            children = node.get("nodes", [])
            if not children:
                return [nid]

            scored = []
            for c in children:
                haystack = " ".join([c.get("title", ""), c.get("summary", "")] + c.get("key_items", [])).lower()
                c_tokens = set(re.findall(r"[a-z0-9]{3,}", haystack))
                scored.append((len(q_tokens & c_tokens), c["node_id"]))
            scored.sort(key=lambda x: x[0], reverse=True)

            picked = [cid for score, cid in scored[:3] if score > 0] or [children[0]["node_id"]]
            out = []
            for cid in picked:
                out.extend(resolve(cid))
            return out

        leaves = []
        for bid in branch_ids:
            leaves.extend(resolve(bid))
        return list(dict.fromkeys(leaves))[:max_leaves]

    def fetch_balanced_pages(
        self,
        node_ids: list[str],
        extra_doc_pages: list[tuple[str, int]],
        expand_provisos: bool = True,
        max_total_pages: int = 8,
    ) -> tuple[str, list[dict]]:
        """Fetch unclipped leaf pages + reserved FTS5 slots, capped at 8 pages."""
        target_pages: list[tuple[str, int, str]] = []
        seen: set[tuple[str, int]] = set()
        tree_budget = max_total_pages - min(len(extra_doc_pages), 4)

        for nid in node_ids:
            node = self.node_map.get(nid)
            if not node:
                continue
            doc_key = nid.split("::")[0]
            s = int(node.get("start_index") or 1)
            raw_e = int(node.get("end_index") or s)
            e = raw_e if not node.get("nodes") else min(raw_e, s + 3)

            if expand_provisos:
                last_text = self.pages.get(doc_key, {}).get(str(e), "")
                if re.search(r"(Provided that|Explanation\.|Illustration)", last_text[-600:], re.I):
                    if str(e + 1) in self.pages.get(doc_key, {}):
                        e += 1

            for p in range(s, e + 1):
                if (doc_key, p) not in seen and len(target_pages) < tree_budget:
                    seen.add((doc_key, p))
                    target_pages.append((doc_key, p, f"tree:{nid}"))

        for doc_key, p in extra_doc_pages:
            if (doc_key, p) not in seen and len(target_pages) < max_total_pages:
                seen.add((doc_key, p))
                target_pages.append((doc_key, p, "fts5_lexical"))

        blocks, citations = [], []
        for doc_key, p, source_tag in target_pages:
            text = self.pages.get(doc_key, {}).get(str(p), "")
            if text:
                citations.append({"doc_key": doc_key, "page": p, "source": source_tag})
                blocks.append(f"=== [{doc_key} | Page {p} | via {source_tag}] ===\n{text}")

        return "\n\n".join(blocks), citations


# ── 3. LangGraph Cyclic State Machine with Ablation Support ──────────────────

AblationMode = Literal["pure_tree", "hybrid_vectorless", "agentic_cyclic"]


class LegalRAGState(TypedDict):
    query: str
    search_query: str
    ablation_mode: AblationMode
    cycle: int
    max_cycles: int
    selected_nodes: list[str]
    retrieved_context: str
    citations: list[dict]
    missing_gaps: list[str]
    grounded: bool
    final_answer: str
    trace: list[str]
    telemetry: list[dict]


def build_fast_vectorless_agent(store: LegalCorpusStore):
    catalog_json = json.dumps(store.build_clean_catalog(), ensure_ascii=False)

    def retrieve(state: LegalRAGState) -> dict:
        cycle = state["cycle"] + 1
        active_q = state["search_query"]
        mode = state["ablation_mode"]
        step_telemetry = list(state["telemetry"])

        if cycle == 1:
            route, t_route = call_router_json(
                "Select up to 3 chapter node_ids across the Indian statutes below that answer the query.\n"
                f'Return JSON: {{"node_ids": ["..."]}}\n\nQuery: {active_q}\n\nCatalog:\n{catalog_json}',
                step_name="cycle_1_route",
                max_tokens=200,
            )
            step_telemetry.append(t_route)
            branches = [n for n in route.get("node_ids", []) if n in store.node_map][:3]
            new_leaves = store.zoom_to_leaf_nodes(branches, active_q, max_leaves=3)
            route_info = f"{t_route['latency_ms']}ms via {t_route['served_by']}"
        else:
            # Cycle 2+: deterministic gap resolution across all Acts using missing section numbers (0 LLM calls)
            new_leaves = []
            route_info = "0ms (deterministic section lookup)"

        merged_nodes = list(dict.fromkeys(new_leaves + state["selected_nodes"]))
        preferred_docs = ({nid.split("::")[0] for nid in merged_nodes} or None) if cycle == 1 else None
        use_hybrid = mode in {"hybrid_vectorless", "agentic_cyclic"}
        fts_hits = store.lexical_search(active_q, preferred_docs=preferred_docs, top_k=4) if use_hybrid else []
        context, citations = store.fetch_balanced_pages(merged_nodes, fts_hits, expand_provisos=use_hybrid)

        msg = f"Cycle {cycle}: nodes={merged_nodes}, fts={fts_hits}, pages={len(citations)} ({route_info})"
        print(" •", msg)
        return {
            "cycle": cycle,
            "selected_nodes": merged_nodes,
            "retrieved_context": context,
            "citations": citations,
            "trace": state["trace"] + [msg],
            "telemetry": step_telemetry,
        }

    def verify_and_synthesise(state: LegalRAGState) -> dict:
        mode = state["ablation_mode"]
        step_telemetry = list(state["telemetry"])

        if not state["retrieved_context"].strip():
            return {
                "grounded": False,
                "final_answer": "INSUFFICIENT_STATUTORY_GROUNDING: No matching statutory text found in corpus.",
                "trace": state["trace"] + ["Verifier: fail-closed (empty context)"],
            }

        if mode == "agentic_cyclic":
            audit, t_ver = call_router_json(
                "Check if the retrieved statutory pages contain enough law to answer the query.\n"
                'Return JSON: {"grounded": true/false, "missing_references": ["Section X"], "followup_query": "Section X keywords"}\n\n'
                f"Query: {state['query']}\n\nRetrieved Pages:\n{state['retrieved_context']}",
                step_name=f"cycle_{state['cycle']}_verify",
                max_tokens=250,
            )
            step_telemetry.append(t_ver)
            grounded = bool(audit.get("grounded", False)) if audit else False
            missing = audit.get("missing_references", [])
            followup = audit.get("followup_query", "")

            if not grounded and state["cycle"] < state["max_cycles"] and (followup or missing):
                gap_q = f"{' '.join(missing)} {followup}".strip()
                msg = f"Verifier: gap {missing} -> retrying '{gap_q}' ({t_ver['latency_ms']}ms)"
                print(" •", msg)
                return {
                    "grounded": False,
                    "missing_gaps": missing,
                    "search_query": gap_q,
                    "trace": state["trace"] + [msg],
                    "telemetry": step_telemetry,
                }

            if not grounded:
                answer, t_syn = call_router_text(
                    "Write a concise fail-closed response starting with 'INSUFFICIENT_STATUTORY_GROUNDING', "
                    f"listing what is missing ({missing}) and summarizing only what is verified below:\n\n"
                    f"Query: {state['query']}\n\nRetrieved Pages:\n{state['retrieved_context']}",
                    step_name=f"cycle_{state['cycle']}_failclosed_synth",
                    max_tokens=1024,
                )
                step_telemetry.append(t_syn)
                return {
                    "grounded": False,
                    "missing_gaps": missing,
                    "final_answer": answer,
                    "trace": state["trace"] + [f"Verifier: fail-closed ({t_syn['latency_ms']}ms)"],
                    "telemetry": step_telemetry,
                }

        answer, t_syn = call_router_text(
            "You are an Indian legal reasoning system. Answer the query concisely and accurately using ONLY the "
            "verbatim statutory excerpts below. Cite exact Act names, Section/Article numbers, and Page numbers.\n\n"
            f"Query: {state['query']}\n\nRetrieved Verbatim Statutory Text:\n{state['retrieved_context']}",
            step_name=f"cycle_{state['cycle']}_synth",
            max_tokens=1200,
        )
        step_telemetry.append(t_syn)
        msg = f"Synthesis complete ({t_syn['latency_ms']}ms via {t_syn['served_by']})"
        print(" •", msg)
        return {
            "grounded": True,
            "missing_gaps": [],
            "final_answer": answer,
            "trace": state["trace"] + [msg],
            "telemetry": step_telemetry,
        }

    def route_after_verify(state: LegalRAGState) -> str:
        if (
            state["ablation_mode"] == "agentic_cyclic"
            and not state["grounded"]
            and state["cycle"] < state["max_cycles"]
            and state.get("missing_gaps")
        ):
            return "retrieve"
        return END

    graph = StateGraph(LegalRAGState)
    graph.add_node("retrieve", retrieve)
    graph.add_node("verify", verify_and_synthesise)
    graph.add_edge(START, "retrieve")
    graph.add_edge("retrieve", "verify")
    graph.add_conditional_edges("verify", route_after_verify, {"retrieve": "retrieve", END: END})
    return graph.compile()


def run_legal_query(
    agent,
    query: str,
    ablation_mode: AblationMode = "agentic_cyclic",
    max_cycles: int = 2,
) -> dict:
    return agent.invoke({
        "query": query,
        "search_query": query,
        "ablation_mode": ablation_mode,
        "cycle": 0,
        "max_cycles": max_cycles if ablation_mode == "agentic_cyclic" else 1,
        "selected_nodes": [],
        "retrieved_context": "",
        "citations": [],
        "missing_gaps": [],
        "grounded": False,
        "final_answer": "",
        "trace": [],
        "telemetry": [],
    })

In [85]:
# Run ONCE to load trees & FTS5 index into memory
store = LegalCorpusStore()
agent = build_fast_vectorless_agent(store)

In [ ]:
q = (
    "Under Indian Constitution what is the punishment for chain-snaching?"
)
out = run_legal_query(agent, q, ablation_mode="agentic_cyclic")
print("\n=== FINAL ANSWER ===\n")
print(out["final_answer"])

In [91]:
q = (
    "Under BNS 2023, what is the punishment for murder, and under what circumstances does the right of private defence of the body extend to causing death?"
)
out = run_legal_query(agent, q, ablation_mode="agentic_cyclic")
print("\n=== FINAL ANSWER ===\n")
print(out["final_answer"])

 • Cycle 1: chapters=['bnsAct2023::0102', 'bnsAct2023::0148'] -> leaves=['bnsAct2023::0149'], fts=[('bnsAct2023', 96), ('bnsAct2023', 97)], pages=7 (3979.6+1354.4ms via gemini-3.5-flash-lite)

=== FINAL ANSWER ===

INSUFFICIENT_STATUTORY_GROUNDING

**Missing:**
- [Section 38 of the BNS 2023 specifying the exact circumstances under which the right of private defence of the body extends to causing death]

**Verified Summary:**
- **Punishment for Murder (Section 103):** Whoever commits murder shall be punished with death or imprisonment for life, and shall also be liable to fine. If committed by a group of five or more persons acting in concert on grounds of race, caste, community, sex, place of birth, language, personal belief, or similar grounds, each member is punishable with death or imprisonment for life and a fine.
- **Punishment for Murder by Life-Convict (Section 104):** A person under a sentence of imprisonment for life who commits murder shall be punished with death or imprisonm